# Proyecto Integrador: API de Ventas con Docker y CI/CD

Este notebook documenta, paso a paso, el proyecto integrador que combina
**Docker**, **contenerización de aplicaciones**, **principios de CI/CD** y
**automatización con GitHub Actions**.

## Objetivo del proyecto

Construir una solución capaz de:

1. Ejecutar pruebas automáticamente.
2. Construir una imagen Docker.
3. Versionar la aplicación.
4. Publicar la imagen en un registro.
5. Preparar el despliegue automático.

## Escenario

Una empresa cuenta con una API en Python para consultar información de
ventas. Cada vez que un desarrollador actualiza el código se ejecutan
pruebas, se valida la sintaxis, se construye una imagen Docker, se publica
y se prepara para despliegue — todo de forma automática.

## Arquitectura general

```
GitHub → GitHub Actions → Pruebas → Docker Build → Docker Registry → Deploy
```


## Estructura del proyecto

```
ventas_api/
├── app.py
├── requirements.txt
├── tests/
│   └── test_app.py
├── Dockerfile
└── .github/
    └── workflows/
        └── pipeline.yml
```

El trabajo está dividido en dos partes iguales:

- **Parte 1** — Aplicación y Contenerización (`app.py`, `requirements.txt`,
  `tests/`, `Dockerfile`).
- **Parte 2** — Pipeline CI/CD (`pipeline.yml` y su documentación).

Este notebook recorre ambas partes.


---
## Parte 1 — Desarrollo de la aplicación

La aplicación es una API Flask con un único endpoint que confirma que el
servicio está operativo. A continuación se define y se ejecuta en vivo
dentro de este notebook.


In [ ]:
from flask import Flask

app = Flask(__name__)


@app.route("/")
def inicio():
    return {
        "mensaje": "API de ventas operativa"
    }


print("App Flask definida correctamente.")


### Dependencias (`requirements.txt`)

```
flask
pytest
```

Estas dos bibliotecas son las únicas necesarias: `flask` para la API y
`pytest` para la prueba automatizada.


### Prueba automatizada (`tests/test_app.py`)

La prueba real del proyecto vive en un archivo separado y se ejecuta con
`pytest` desde la terminal o el pipeline:

```python
from app import app

def test_inicio():
    cliente = app.test_client()
    respuesta = cliente.get("/")
    assert respuesta.status_code == 200
```

Aquí abajo se ejecuta la misma verificación directamente en el notebook,
a modo de comprobación rápida.


In [ ]:
def test_inicio():
    cliente = app.test_client()
    respuesta = cliente.get("/")
    assert respuesta.status_code == 200
    return respuesta.get_json()


resultado = test_inicio()
print("Prueba pasada. Respuesta:", resultado)


---
## Parte 2 — Contenerización con Docker

Una vez verificada la app, se empaqueta en una imagen Docker.

### Dockerfile

```dockerfile
FROM python:3.12

WORKDIR /app

COPY requirements.txt .

RUN pip install -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["python", "app.py"]
```

### Construcción y verificación local (fuera del notebook, en terminal)

```bash
docker build -t ventas-api .
docker run -p 5000:5000 ventas-api
```

Acceder a `http://localhost:5000` debe devolver:

```json
{
  "mensaje": "API de ventas operativa"
}
```


---
## Parte 3 — Pipeline CI/CD con GitHub Actions

El workflow automatiza todo lo anterior cada vez que hay un `push`.

### Workflow completo (`.github/workflows/pipeline.yml`)

```yaml
name: CI/CD Pipeline

on:
  push:

jobs:

  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - name: Instalar Dependencias
        run: |
          pip install -r requirements.txt
      - name: Ejecutar Pruebas
        run: |
          pytest
      - name: Validar Código
        run: |
          python -m py_compile app.py

  build:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: docker/setup-buildx-action@v3
      - uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_PASSWORD }}
      - uses: docker/build-push-action@v6
        with:
          context: .
          push: true
          tags: usuario/ventas-api:latest
```

### Qué hace cada job

| Job | Depende de | Responsabilidad |
|---|---|---|
| `test` | — | Instala dependencias, corre `pytest` y valida sintaxis con `py_compile`. Si falla, el pipeline se detiene. |
| `build` | `test` | Construye la imagen Docker y la publica en Docker Hub usando `secrets.DOCKER_USERNAME` / `secrets.DOCKER_PASSWORD`. |

### Flujo completo

```
Git Push → Checkout → Python Setup → Dependencias → Pruebas
         → Validación → Docker Build → Docker Hub
```


---
## Buenas prácticas aplicadas

- **Automatizar validaciones**: ninguna versión se publica sin pasar
  pruebas y validación de sintaxis primero.
- **Dockerfile simple**: una sola etapa, sin pasos innecesarios.
- **Uso de secretos**: las credenciales nunca están en el código.
- **Pipeline modular**: dos jobs separados y encadenados con `needs`.

## Mejoras posibles

| Mejora | Herramienta |
|---|---|
| Cobertura de pruebas | `pytest --cov` |
| Escaneo de seguridad | Bandit |
| Análisis de calidad | SonarQube |
| Versionamiento | Tags semánticos (`v1.0.0`, `v1.1.0`) en vez de solo `latest` |
| Despliegue automático | Etapa adicional Docker Registry → Producción |


## La misma arquitectura en ingeniería de datos

El patrón Checkout → Pruebas → Docker Build → Registro → Deploy no es
exclusivo de APIs; aplica igual a:

- **Procesos ETL**: script Python → Docker → CI/CD.
- **APIs analíticas**: FastAPI → Docker → GitHub Actions.
- **Procesamiento de datos**: Python + Pandas → Docker → CI/CD.
- **Automatización de reportes**: scripts → Docker → pipeline.

## Conclusiones

Este proyecto integrador reúne Docker, contenedores, CI/CD y GitHub
Actions en un único flujo automatizado: validar código, construir
imágenes Docker y preparar despliegues reproducibles y controlados —
el mismo ciclo que usan las organizaciones modernas para automatizar
desarrollo y entrega de software.
